In [1]:
import glob
import numpy as np
import pandas as pd
import os
import matplotlib.pylab as plt
from tqdm import tqdm
import seaborn as sns
import scipy.stats as stats
from pybaselines import Baseline
from pybaselines.utils import gaussian
import csv
import numpy as np
import pandas as pd
import scipy.stats as stats
import scipy.signal as signal
import matplotlib.pylab as plt
import seaborn as sns
from pybaselines import Baseline
from pybaselines.utils import gaussian
from sklearn.preprocessing import MinMaxScaler
import matplotlib.image as mpimg
#from read_roi import read_roi_zip
import math
import matplotlib.colors as mcolors
from tqdm import tqdm
import matplotlib
from read_roi import read_roi_zip
matplotlib.use('Agg')



In [2]:
# Define the root directory
root_dir = r"C:\Users\wiesbrock\Desktop\Projekte\AMH\aktuell\ROIs_Traces_for_Christopher\Traces" 

# Lists to store the paths for RawGre and RawRed files
raw_gre_paths = []
raw_red_paths = []
roi_files=[]

for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith('.zip'):
            # Create the full path
            full_path = os.path.join(dirpath, filename)
            roi_files.append(full_path)

# Walk through the directory
for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith('.csv'):
            # Create the full path
            full_path = os.path.join(dirpath, filename)
            if "RawGre" in filename:
                raw_gre_paths.append(full_path)
            elif "RawRed" in filename:
                raw_red_paths.append(full_path)
   
print(roi_files)

def crosscorr(datax, datay, lag=0):
    """ Lag-N cross correlation. 
    Parameters
    ----------
    lag : int, default 0
    datax, datay : pandas.Series objects of equal length
    Returns
    ----------
    crosscorr : float
    """
    return datax.corr(datay.shift(lag))

def berechne_flaeche(x_array, y_array):
    n = len(x_array)  # Anzahl der Punkte im Polygon
    flaeche = 0
    
    # Anwenden der Shoelace-Formel
    for p in range(n):
        q = (p + 1) % n  # Der nächste Punkt (Index i+1, aber modulo n für den letzten Punkt)
        flaeche += x_array[p] * y_array[q]
        flaeche -= y_array[p] * x_array[q]
    
    flaeche = abs(flaeche) / 2.0
    return flaeche

list_excel=pd.DataFrame(raw_gre_paths)
list_excel.to_excel(r'C:\Users\wiesbrock\Desktop\Projekte\heatmap_liste.xlsx')

from read_roi import read_roi_zip

import numpy as np
from scipy.spatial.distance import cdist

# Funktion zur Berechnung des Centroids (Schwerpunkts) eines ROIs
def calculate_centroid(roi):
    # Annahme: ROI hat Eckpunkte in 'x' und 'y'-Koordinaten als Listen
    x_coords = roi['x']
    y_coords = roi['y']
    
    # Centroid-Berechnung: Mittelwert der x- und y-Koordinaten
    centroid_x = np.mean(x_coords)
    centroid_y = np.mean(y_coords)
    
    return np.array([centroid_x, centroid_y])

['C:\\Users\\wiesbrock\\Desktop\\Projekte\\AMH\\aktuell\\ROIs_Traces_for_Christopher\\Traces\\Adult_Control\\Post-early\\DC_deec20211213_PrintedInvivo_AMH_GCamP_23W_TBNr192_Control_2P_StagNew_TemVar-Series003.tif.zip', 'C:\\Users\\wiesbrock\\Desktop\\Projekte\\AMH\\aktuell\\ROIs_Traces_for_Christopher\\Traces\\Adult_Control\\Post-early\\DC_deec20211213_PrintedInvivo_AMH_GCamP_23W_TBNr192_Control_2P_StagNew_TemVar-Series004.tif.zip', 'C:\\Users\\wiesbrock\\Desktop\\Projekte\\AMH\\aktuell\\ROIs_Traces_for_Christopher\\Traces\\Adult_Control\\Post-early\\DC_deec20211213_PrintedInvivo_AMH_GCamP_23W_TBNr192_Control_2P_StagNew_TemVar-Series005part1.tif.zip', 'C:\\Users\\wiesbrock\\Desktop\\Projekte\\AMH\\aktuell\\ROIs_Traces_for_Christopher\\Traces\\Adult_Control\\Post-early\\DC_deec20211213_PrintedInvivo_AMH_GCamP_23W_TBNr192_Control_2P_StagNew_TemVar-Series005part2.tif.zip', 'C:\\Users\\wiesbrock\\Desktop\\Projekte\\AMH\\aktuell\\ROIs_Traces_for_Christopher\\Traces\\Adult_Control\\Post-earl

In [3]:
from scipy.signal import savgol_filter

window_length = 11  # Fenstergröße für den Filter (muss ungerade sein)
polyorder = 3


corr_list=[]
corr_list_inter=[]
corr_list_intra=[]
area_1_l=[]
area_2_l=[]
distance_l=[]
name_1=[]
name_2=[]
norm_name_1=[]
norm_name_2=[]
file_list=[]


z=-1
roi=-1
for i in tqdm(range(len(raw_gre_paths))):
    
    z=z+1
    rois = read_roi_zip(roi_files[i])
    df=pd.read_csv(raw_gre_paths[i])
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    names=df.columns
    names = [name for name in df.columns if ('G0' in name or 'G2' in name) and 'G0_-U' not in name]
    filtered_data = {key: value for key, value in rois.items() if 'G0' in key or 'G2' in key}
    filtered_data = {key: value for key, value in filtered_data.items() if 'G0_-U' not in key}
    filtered_data=pd.DataFrame(filtered_data)
    filtered_names=pd.DataFrame(filtered_data)
    indices=filtered_names.columns
    x_array=filtered_data.iloc[1]
    y_array=filtered_data.iloc[2]
    df=df[names]
    tubulus_list=[]
 
    df=df[names]
    tubulus_list=[]
   
    for t in range(len(names)):
        tubulus_list.append(names[t][9:11])
        
    mask = np.zeros((len(df.columns), len(df.columns)), dtype=bool)
    label = tubulus_list
    
    for m in range(len(mask)):
        for n in range(len(mask)):  
            if label[m]==label[n]:
                mask[m, n] = 1
                
    df=np.array(df)
    
    max_diff=10
    cross_corr_array=np.zeros((len(names),len(names)))
    lag_array=np.zeros((len(names),len(names)))


    


    for k in range(len(names)):
        for l in range(len(names)):
            roi=roi+1
            trace_1=(df[:,k])
            roi_name_1=indices[k]
            trace_2=(df[:,l])
            roi_name_2=indices[l]
            init_trace_1=(df[:,k])
            init_trace_2=(df[:,l])
            
            trace_1 = pd.Series(trace_1).fillna(0)
            trace_2 = pd.Series(trace_2).fillna(0)
            trace_1[trace_2==0]=0
            trace_2[trace_1==0]=0
            
            trace_1=trace_1[trace_1!=0]
            trace_2=trace_2[trace_2!=0]
            
            if len(trace_1)>10:
                if len(trace_2)>10:
                    x = np.linspace(0, len(trace_1), len(trace_1))
                    baseline_fitter = Baseline(x_data=x)

                    trace_1 = trace_1-baseline_fitter.noise_median(trace_1)[0]
                    x = np.linspace(0, len(trace_2), len(trace_2))
                    baseline_fitter = Baseline(x_data=x)

                    trace_2 =trace_2- baseline_fitter.noise_median(trace_2)[0]
                    
                    #trace_1 = savgol_filter(trace_1, window_length, polyorder)
                    #trace_2 = savgol_filter(trace_2, window_length, polyorder)


                    trace_1=pd.DataFrame(trace_1)
                    trace_2=pd.DataFrame(trace_2)
                    trace_1=trace_1.squeeze()
                    trace_2=trace_2.squeeze()
                    cross_list=np.zeros((max_diff))
                    for m in range(max_diff):
                        cross_list[m]=crosscorr(trace_1,trace_2,lag=m)
                    #if len(np.where(cross_list==np.max(cross_list))[0])==1:
                    #    lag_array[k,l]=np.where(cross_list==np.max(cross_list))[0]
                    #else:
                    #    lag_array[k,l]=np.where(cross_list==np.max(cross_list))[0][0]
                    cross_corr_array[k,l]=np.max(cross_list)
                    
                    x_1=np.mean(x_array[k])
                    x_2=np.mean(x_array[l])

                    y_1=np.mean(y_array[k])
                    y_2=np.mean(y_array[l])

                    distance = np.sqrt((x_2 - x_1)**2 + (y_2 - y_1)**2)

                    area_1=berechne_flaeche(x_array[k],y_array[k])
                    area_2=berechne_flaeche(x_array[l],y_array[l])

                    name_1.append(indices[k])
                    name_2.append(indices[l])
                    distance_l.append(distance)
                    area_1_l.append(area_1)
                    area_2_l.append(area_2)
                    norm_name_1.append(names[k])
                    norm_name_2.append(names[l])
                    file_list.append(raw_gre_paths[i])
                    
                    #plt.figure()
                    #plt.title(str(cross_corr_array[k,l]))
                    #plt.subplot(411)
                    #plt.plot(trace_1)
                    #plt.subplot(412)
                    #plt.plot(init_trace_1)
                    #plt.subplot(413)
                    #plt.plot(trace_2)
                    #plt.subplot(414)
                    #plt.plot(init_trace_2)
                    #plt.savefig(r'C:\Users\wiesbrock\Desktop\Projekte\AMH\Figures'+'/'+str(names[k])+str(names[l]))
                    #plt.close('all')
                    #roi_list.append(roi)
                    corr_list.append(cross_corr_array[k,l])
        
    cross_corr_array_intra=cross_corr_array[mask==1]
    cross_corr_array_inter=cross_corr_array[mask==0]
        
    corr_list_inter.append(cross_corr_array_inter[cross_corr_array_inter<0.99])    
    corr_list_intra.append(cross_corr_array_intra[cross_corr_array_intra<0.99])  
    #plt.figure(dpi=300)
    
    #plt.title(str(raw_gre_paths[i]))
    #plt.imshow(cross_corr_array, cmap='YlOrRd',vmin=-0.3,vmax=1)
    
    #plt.colorbar()
    #plt.savefig(r'C:\Users\wiesbrock\Desktop\Projekte\AMH\Figures\Heatmaps'+'/'+str(z)+'.png')
    #plt.close('all')

all_list=pd.DataFrame()
all_list['name_1']=name_1
all_list['name_2']=name_2
all_list['norm_name_1']=norm_name_1
all_list['norm_name_2']=norm_name_2
all_list['area_1']=area_1_l
all_list['area_2']=area_2_l
all_list['distance']=distance_l
all_list['file']=file_list

all_list.to_excel(r'C:\Users\wiesbrock\Desktop\Projekte\AMH\Figures\all_list.xlsx')


  6%|████▌                                                                             | 6/108 [01:12<20:37, 12.13s/it]


KeyboardInterrupt: 

In [8]:
liste_files=pd.DataFrame(file_list)
print(liste_files.head())
liste_files.to_excel(r'C:\Users\wiesbrock\Desktop\Projekte\AMH\Figures\file_list.xlsx')

                                                   0
0  C:\Users\wiesbrock\Desktop\Projekte\AMH\aktuel...
1  C:\Users\wiesbrock\Desktop\Projekte\AMH\aktuel...
2  C:\Users\wiesbrock\Desktop\Projekte\AMH\aktuel...
3  C:\Users\wiesbrock\Desktop\Projekte\AMH\aktuel...
4  C:\Users\wiesbrock\Desktop\Projekte\AMH\aktuel...


In [9]:
import itertools

data_1 = list(itertools.chain(*corr_list_intra))
data_2=list(itertools.chain(*corr_list_inter))
print(len(data_1))
print(len(data_2))
data_1=np.array(data_1)
data_2=np.array(data_2)

#data_1=data_1[data_1<0.5]
#data_2=data_2[data_2<0.5]

data_1=data_1[data_1>0.]
data_2=data_2[data_2>0.]

print(np.max(data_1))
print(np.max(data_2))




data=data_1,data_2



plt.figure(dpi=300)
sns.histplot(data_1, stat='percent',label='intra')
sns.histplot(data_2,stat='percent', label='inter')
plt.legend()

plt.figure(dpi=300)
sns.violinplot(data=data, cut=0)


print(len(data_1))
print(len(data_2))

import scipy.stats as stats
print(stats.ttest_ind(data_1,data_2))

print((np.mean(data_1)-np.mean(data_2))/np.std(data_1))

251814
128914
0.9891379661770193
0.9149569814005345
221214
106691
Ttest_indResult(statistic=54.987392774944986, pvalue=0.0)
0.20041359603562311


In [10]:
print(len(corr_list))
corr_list=pd.DataFrame(corr_list)
corr_list.to_excel(r'C:\Users\wiesbrock\Desktop\Projekte\AMH\Figures\corr.xlsx')



385257


In [11]:
all_list=pd.DataFrame()
all_list['name_1']=name_1
all_list['name_2']=name_2
all_list['norm_name_1']=norm_name_1
all_list['norm_name_2']=norm_name_2
all_list['area_1']=area_1_l
all_list['area_2']=area_2_l
all_list['distance']=distance_l
all_list['correlation']=corr_list

all_list.to_excel(r'C:\Users\wiesbrock\Desktop\Projekte\AMH\Figures\all_list.xlsx')

In [12]:
#raw_gre_path
# Lists to store the paths for RawGre and RawRed files
raw_gre_paths = []
raw_red_paths = []
roi_files=[]
corr_list=[]

for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith('.zip'):
            # Create the full path
            full_path = os.path.join(dirpath, filename)
            roi_files.append(full_path)

# Walk through the directory
for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith('.csv'):
            # Create the full path
            full_path = os.path.join(dirpath, filename)
            if "RawGre" in filename:
                raw_gre_paths.append(full_path)
            elif "RawRed" in filename:
                raw_red_paths.append(full_path)
   
print(roi_files)
corr_list=[]
from scipy.signal import savgol_filter
window_length = 11  # Fenstergröße für den Filter (muss ungerade sein)
polyorder = 3
max_diff=12
for i in tqdm(range(2000)):
    rng=np.random.randint(0,len(raw_gre_paths),1)
    rng=int(rng)
    df=pd.read_csv(raw_gre_paths[rng])
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    names=df.columns
    rng=np.random.randint(0,len(names),1)
    rng=int(rng)
    trace_1=df[names[rng]]
    
    rng=np.random.randint(0,len(raw_gre_paths),1)
    rng=int(rng)
    df=pd.read_csv(raw_gre_paths[rng])
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    names=df.columns
    rng=np.random.randint(0,len(names),1)
    rng=int(rng)
    trace_2=df[names[rng]]
    trace_1 = pd.Series(trace_1).fillna(0)
    trace_2 = pd.Series(trace_2).fillna(0)
    #trace_1[trace_2==0]=0
    #trace_2[trace_1==0]=0

    #trace_1=trace_1[trace_1!=0]
    #trace_2=trace_2[trace_2!=0]

    if len(trace_1)>10:
        if len(trace_2)>10:
            x = np.linspace(0, len(trace_1), len(trace_1))
            baseline_fitter = Baseline(x_data=x)

            trace_1 = trace_1-baseline_fitter.noise_median(trace_1)[0]
            x = np.linspace(0, len(trace_2), len(trace_2))
            baseline_fitter = Baseline(x_data=x)

            trace_2 =trace_2- baseline_fitter.noise_median(trace_2)[0]

            trace_1 = savgol_filter(trace_1, window_length, polyorder)
            trace_2 = savgol_filter(trace_2, window_length, polyorder)


            trace_1=pd.DataFrame(trace_1)
            trace_2=pd.DataFrame(trace_2)
            trace_1=trace_1.squeeze()
            trace_2=trace_2.squeeze()
            cross_list=np.zeros((max_diff))
            for m in range(max_diff):
                cross_list[m]=crosscorr(trace_1,trace_2,lag=m)
            #if len(np.where(cross_list==np.max(cross_list))[0])==1:
            #    lag_array[k,l]=np.where(cross_list==np.max(cross_list))[0]
            #else:
            #    lag_array[k,l]=np.where(cross_list==np.max(cross_list))[0][0]
            corr_list.append(np.max(cross_list))

            


['C:\\Users\\wiesbrock\\Desktop\\Projekte\\AMH\\aktuell\\ROIs_Traces_for_Christopher\\Traces\\Adult_Control\\Post-early\\DC_deec20211213_PrintedInvivo_AMH_GCamP_23W_TBNr192_Control_2P_StagNew_TemVar-Series003.tif.zip', 'C:\\Users\\wiesbrock\\Desktop\\Projekte\\AMH\\aktuell\\ROIs_Traces_for_Christopher\\Traces\\Adult_Control\\Post-early\\DC_deec20211213_PrintedInvivo_AMH_GCamP_23W_TBNr192_Control_2P_StagNew_TemVar-Series004.tif.zip', 'C:\\Users\\wiesbrock\\Desktop\\Projekte\\AMH\\aktuell\\ROIs_Traces_for_Christopher\\Traces\\Adult_Control\\Post-early\\DC_deec20211213_PrintedInvivo_AMH_GCamP_23W_TBNr192_Control_2P_StagNew_TemVar-Series005part1.tif.zip', 'C:\\Users\\wiesbrock\\Desktop\\Projekte\\AMH\\aktuell\\ROIs_Traces_for_Christopher\\Traces\\Adult_Control\\Post-early\\DC_deec20211213_PrintedInvivo_AMH_GCamP_23W_TBNr192_Control_2P_StagNew_TemVar-Series005part2.tif.zip', 'C:\\Users\\wiesbrock\\Desktop\\Projekte\\AMH\\aktuell\\ROIs_Traces_for_Christopher\\Traces\\Adult_Control\\Post-earl

100%|██████████████████████████████████████████████████████████████████████████████| 2000/2000 [40:37<00:00,  1.22s/it]


In [13]:
corr_list=pd.DataFrame(corr_list)
corr_list.to_excel(r'C:\Users\wiesbrock\Desktop\Projekte\AMH\Figures\sramble.xlsx')